## Loading dataset

In [62]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
import scipy.stats as stats
folder = '/content/drive/MyDrive/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [63]:
df = pd.read_csv(folder + 'room_based_shelters.csv')
df['OCCUPANCY_DATE'] = pd.to_datetime(df['OCCUPANCY_DATE'], errors='coerce')

In [64]:
sorted(df['OCCUPANCY_DATE'].dt.year.unique())

[np.int32(2021), np.int32(2022), np.int32(2024), np.int32(2025)]

In [65]:
df.columns

Index(['_id', 'OCCUPANCY_DATE', 'SHELTER_ID', 'LOCATION_ID',
       'LOCATION_ADDRESS', 'LOCATION_POSTAL_CODE', 'SECTOR', 'PROGRAM_AREA',
       'SERVICE_USER_COUNT', 'CAPACITY_TYPE', 'CAPACITY_ACTUAL_ROOM',
       'CAPACITY_FUNDING_ROOM', 'OCCUPIED_ROOMS', 'UNOCCUPIED_ROOMS',
       'UNAVAILABLE_ROOMS', 'OCCUPANCY_RATE_ROOMS'],
      dtype='object')

## Setting up features + splitting

In [66]:
# get time features set up
df['year'] = df['OCCUPANCY_DATE'].dt.year
df['month'] = df['OCCUPANCY_DATE'].dt.month
df['dayofweek'] = df['OCCUPANCY_DATE'].dt.dayofweek

In [67]:
# build features + target
df['pressure'] = df['OCCUPIED_ROOMS'] / df['CAPACITY_ACTUAL_ROOM']

# high pressure will be 1 and not high pressure (low pressure/normal) will be 0
# high pressure -> how full the shelther is
df['high_pressure'] = (df['pressure'] > 0.9).astype(int)

# sort data in time orders per shelther
df = df.sort_values(['SHELTER_ID', 'OCCUPANCY_DATE'])

# create the future target (each shelter is treated differently)
# future pressure means the next 2 week's pressure status... we are doing lagged target forecasting
df['future_pressure'] = df.groupby('SHELTER_ID')['high_pressure'].shift(-30)

df_model = df.dropna(subset=['future_pressure']).copy()

In [68]:
# features and target
X = df_model[[
    'SHELTER_ID',
    'SERVICE_USER_COUNT',
    'CAPACITY_ACTUAL_ROOM',
    'CAPACITY_FUNDING_ROOM',
    'year',
    'month',
    'dayofweek'
]]

y = df_model['future_pressure']

In [69]:
# split the data
train = df_model[df_model['OCCUPANCY_DATE'] < '2025-01-01']
test = df_model[df_model['OCCUPANCY_DATE'] >= '2025-01-01']

X_train = train[X.columns]
y_train = train['future_pressure']

X_test = test[X.columns]
y_test = test['future_pressure']


## Cross validation for learning rate

In [70]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
import numpy as np

In [71]:
learning_rates = [0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 0.4]

In [72]:
tscv = TimeSeriesSplit(n_splits=5)

In [73]:
results = {}

for lr in learning_rates:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=lr,
            max_depth=6,
            max_iter=200,
            class_weight="balanced"
        )

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        score = f1_score(y_val, preds, pos_label=1)
        fold_scores.append(score)

    results[lr] = np.mean(fold_scores)

print(results)
print("Best learning rate:", max(results, key=results.get))

{0.01: np.float64(0.9056079187721954), 0.03: np.float64(0.9247869588681205), 0.05: np.float64(0.9299017247715817), 0.1: np.float64(0.9375328778454133), 0.2: np.float64(0.9384848889281848), 0.3: np.float64(0.9375724734700592), 0.4: np.float64(0.9370403535896992)}
Best learning rate: 0.2


## Cross validation for max depth

In [74]:
depths = [3, 4, 5, 6, 7, 8, 10]

tscv = TimeSeriesSplit(n_splits=5)

results = {}

for depth in depths:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=0.01,
            max_depth=depth,
            max_iter=200
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        fold_scores.append(f1_score(y_val, preds))

    results[depth] = np.mean(fold_scores)

best_depth = max(results, key=results.get)
print(best_depth)

7


## Cross validation for max iterations

In [75]:
# values to test
iters = [100, 200, 500, 1000]

tscv = TimeSeriesSplit(n_splits=5)

results = {}

for n_iter in iters:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=0.01,
            max_depth=6,
            max_iter=n_iter
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        score = f1_score(y_val, preds)
        fold_scores.append(score)

    results[n_iter] = np.mean(fold_scores)

print(results)

best_iter = max(results, key=results.get)

print("Best max_iter:", best_iter)

{100: np.float64(0.9324904302562548), 200: np.float64(0.9323432590835719), 500: np.float64(0.9386975953562653), 1000: np.float64(0.9412851780259912)}
Best max_iter: 1000


## Training the model + Results

In [76]:
# train the model
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report

train = df_model[df_model['OCCUPANCY_DATE'] < '2025-01-01']
test = df_model[df_model['OCCUPANCY_DATE'] >= '2025-01-01']

X_train = train[X.columns]
y_train = train['future_pressure']

X_test = test[X.columns]
y_test = test['future_pressure']

model = HistGradientBoostingClassifier(
    max_depth=3,
    learning_rate=0.2,
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

         0.0       0.05      0.03      0.03       159
         1.0       0.97      0.99      0.98      5326

    accuracy                           0.96      5485
   macro avg       0.51      0.51      0.51      5485
weighted avg       0.94      0.96      0.95      5485



In [77]:
# when the model says not high pressure it's right 5% of the time
  # recall: 0.03... it only catches 3% of all true “normal” cases.. it misses a lot of normal days.
  # performance of a classification model -> F1-score: 0.03 (overall very weak performacne for the not high pressure class)
  # model is not good at identifying “safe / low-pressure” situations

# when the model says high pressure it's right 97% of the time
  # recall: 0.99... it only catches 99% of all true high pressure cases
  # performance of a classification model -> F1-score: 0.98 ( strong performance)
  # the model is excellent at detecting shelter stress/overload

# OVERALL: this model is a very strong early warning system for shelter overload, NOT a balanced classifier for both normal and high pressure states
  # accuracy -> f1-score: 0.96 (our data is biased because we have a lot more samples of high pressure classes)
  # macro avg: taking the average performance of each class separately
  # weighted avg: takes the number of samples (support #) in each class into account

In [78]:
# most shelter day observations in the test period were already operating above the 90% pressure threshold
# 5326/5485 ~= 97% of our test data is high pressure.

## Shelter pressure Ranking (30-day outlook)

In [79]:
# set up model probabilites
df_model['risk_prob'] = model.predict_proba(X)[:, 1]

In [80]:
# aggregate by shelter
  # avg_future_risk: overall long-term stress level
  # max_future_risk: captures spikes
  # high_risk_days: % of danger periods (0.7 -> 70% of days are high pressure)
  # total_days: how much data we have per shelter

shelter_risk = df_model.groupby('SHELTER_ID').agg(
    avg_future_risk=('risk_prob', 'mean'),
    max_future_risk=('risk_prob', 'max'),
    high_risk_days=('risk_prob', lambda x: (x > 0.8).mean()),
    total_days=('risk_prob', 'count')
).reset_index()

In [81]:
# rank the shelters by risk
top_risk_shelters = shelter_risk.sort_values(
    'avg_future_risk',
    ascending=False
)

In [82]:
# add location + sector info
locations = df[['SHELTER_ID', 'LOCATION_ADDRESS', 'LOCATION_POSTAL_CODE', 'SECTOR']].drop_duplicates()

top_risk_shelters = top_risk_shelters.merge(
    locations,
    on='SHELTER_ID',
    how='left'
)

## Mapping shelter risks

In [83]:
!pip install geopy

In [84]:
df['FULL_ADDRESS'] = (df['LOCATION_ADDRESS'].astype(str) + ", Toronto, Canada")

In [85]:
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="shelter_map")

coords = {}

unique_addresses = df[['SHELTER_ID', 'FULL_ADDRESS']].drop_duplicates()

for _, row in unique_addresses.iterrows():
    shelter_id = row['SHELTER_ID']
    address = row['FULL_ADDRESS']

    try:
        location = geolocator.geocode(address)
        if location:
            coords[shelter_id] = (location.latitude, location.longitude)
        else:
            coords[shelter_id] = (None, None)
    except:
        coords[shelter_id] = (None, None)

    time.sleep(1)  # important to avoid blocking

In [86]:
coord_df = pd.DataFrame.from_dict(coords, orient='index', columns=['lat', 'lon'])
coord_df['SHELTER_ID'] = coord_df.index
coord_df.reset_index(drop=True, inplace=True)

In [87]:
map_df = top_risk_shelters.merge(coord_df, on='SHELTER_ID', how='left')
map_df = map_df.dropna(subset=['lat', 'lon'])

In [88]:
import folium

toronto_map = folium.Map(location=[43.7, -79.4], zoom_start=11)

In [89]:
def color(r):
    if r > 0.85:
        return 'red'
    elif r > 0.7:
        return 'orange'
    else:
        return 'green'

In [90]:
for _, row in map_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        color=color(row['avg_future_risk']),
        fill=True,
        fill_opacity=0.7,
        popup=f"{row['LOCATION_ADDRESS']} | Risk: {row['avg_future_risk']:.2f}"
    ).add_to(toronto_map)

toronto_map

In [91]:
df['SHELTER_ID'].nunique()

23